# Historical workflow (redacted, read-only)

This is an archival view of the original AWS JupyterLab notebook **cell sources**, in original order. Individual cells are rendered as Markdown rather than executable Python to prevent accidental re-running of paid SageMaker jobs, S3 writes or Pipeline updates. All execution outputs, timestamps and notebook-specific metadata were removed; account, bucket, role and personal run identifiers were redacted. This is **not** a verbatim executable notebook. Use the clean reproduction notebook or `scripts/` instead.


### Original cell 01

```python

from pathlib import Path

Path("src").mkdir(exist_ok=True)

print("Source directory:", Path("src").resolve())
```

### Original cell 02

```python
%%writefile src/preprocess.py
from pathlib import Path

import os
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = Path(
    os.environ.get("PROCESSING_BASE", "/opt/ml/processing")
)
INPUT_PATH = BASE / "input" / "breast_cancer.csv"

# Load dataset
df = pd.read_csv(INPUT_PATH)

if "target" not in df.columns:
    raise ValueError("Missing target column")

# 0 = malignant, 1 = benign
print("Total samples:", len(df))
print("Class distribution:")
print(df["target"].value_counts().sort_index())

# Stratified 80:20 split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"],
)

# Save outputs for downstream pipeline steps
for folder, data, filename in [
    ("train", train_df, "train.csv"),
    ("test", test_df, "test.csv"),
]:
    output_dir = BASE / folder
    output_dir.mkdir(parents=True, exist_ok=True)
    data.to_csv(output_dir / filename, index=False)

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))
print("Preprocessing completed")
```

### Original cell 03

```python

from pathlib import Path

script = Path("src/preprocess.py")

print("File exists:", script.is_file())
print("File size:", script.stat().st_size, "bytes")
```

### Original cell 04

```python

from pathlib import Path
import boto3
from sklearn.datasets import load_breast_cancer

# 1. 데이터셋 로드
df = load_breast_cancer(as_frame=True).frame

# 2. 데이터 검증
assert df.shape == (569, 31)
assert df["target"].value_counts().to_dict() == {1: 357, 0: 212}

print("Dataset shape:", df.shape)
print("Features:", df.shape[1] - 1)
print("Target distribution:")
print(df["target"].value_counts().sort_index())

# 3. 로컬 CSV 저장
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "breast_cancer.csv"
df.to_csv(csv_path, index=False)

print("\nLocal CSV:", csv_path)

# 4. 프로젝트 전용 S3에 업로드
bucket = "REDACTED_BUCKET"
key = "breast-cancer-pipeline/input/breast_cancer.csv"

s3 = boto3.client("s3", region_name="ap-northeast-2")
s3.upload_file(str(csv_path), bucket, key)

# 5. 업로드 검증
response = s3.head_object(Bucket=bucket, Key=key)

print("S3 upload: OK")
print("S3 object size:", response["ContentLength"], "bytes")
print("S3 URI:", f"s3://{bucket}/{key}")
```

### Original cell 05

```python

from pathlib import Path
import os
import shutil
import subprocess
import sys
import pandas as pd

# 테스트 전용 경로 준비
test_base = Path("local_test").resolve()
input_dir = test_base / "input"
input_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    "data/breast_cancer.csv",
    input_dir / "breast_cancer.csv"
)

# SageMaker Processing 경로를 로컬 테스트 경로로 변경
env = os.environ.copy()
env["PROCESSING_BASE"] = str(test_base)

result = subprocess.run(
    [sys.executable, "src/preprocess.py"],
    env=env,
    capture_output=True,
    text=True,
    check=True,
)

print("=== Preprocessing Output ===")
print(result.stdout)

# 결과 검증
train = pd.read_csv(test_base / "train" / "train.csv")
test = pd.read_csv(test_base / "test" / "test.csv")

print("=== Validation ===")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain target distribution:")
print(train["target"].value_counts().sort_index())

print("\nTest target distribution:")
print(test["target"].value_counts().sort_index())

assert train.shape == (455, 31)
assert test.shape == (114, 31)
assert len(train) + len(test) == 569

print("\nPreprocessing validation: OK")
```

### Original cell 06

```python

%%writefile src/train.py
import argparse
import os
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--n_estimators",
        type=int,
        default=100,
    )

    parser.add_argument(
        "--random_state",
        type=int,
        default=42,
    )

    args = parser.parse_args()

    # SageMaker Training 환경 변수
    train_dir = Path(
        os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train")
    )
    model_dir = Path(
        os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    )

    # 학습 데이터 로드
    df = pd.read_csv(train_dir / "train.csv")

    X = df.drop(columns=["target"])
    y = df["target"]

    # Random Forest 모델 정의
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        random_state=args.random_state,
        n_jobs=-1,
    )

    # 모델 학습
    model.fit(X, y)

    # 모델 아티팩트 저장
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / "model.joblib"

    joblib.dump(model, model_path)

    print("=== Training Completed ===")
    print("Training samples:", len(df))
    print("Number of features:", X.shape[1])
    print("n_estimators:", args.n_estimators)
    print("random_state:", args.random_state)
    print("Model saved:", model_path)


if __name__ == "__main__":
    main()
```

### Original cell 07

```python

import os
import subprocess
import sys
from pathlib import Path

# 앞 단계에서 생성한 학습 데이터 재사용
train_dir = Path("local_test/train").resolve()
model_dir = Path("local_test/model").resolve()

assert (train_dir / "train.csv").exists()

env = os.environ.copy()
env["SM_CHANNEL_TRAIN"] = str(train_dir)
env["SM_MODEL_DIR"] = str(model_dir)

# Random Forest 학습 실행
result = subprocess.run(
    [
        sys.executable,
        "src/train.py",
        "--n_estimators", "10",
        "--random_state", "42",
    ],
    env=env,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

# 모델 저장 검증
import joblib

model_path = model_dir / "model.joblib"

assert model_path.exists()

model = joblib.load(model_path)

assert model.n_estimators == 10
assert model.n_features_in_ == 30

print("\n=== Validation ===")
print("Model exists:", model_path.exists())
print("Model type:", type(model).__name__)
print("Trees:", model.n_estimators)
print("Features:", model.n_features_in_)
print("Training validation: OK")
```

### Original cell 08

```python

%%writefile src/evaluate.py
import json
import os
import tarfile
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

BASE = Path(
    os.environ.get("PROCESSING_BASE", "/opt/ml/processing")
)

model_archive = BASE / "model" / "model.tar.gz"
test_path = BASE / "test" / "test.csv"
output_dir = BASE / "evaluation"

# Load the model produced by our own training job.
# Do not load model artifacts from untrusted sources.
with tarfile.open(model_archive, "r:gz") as tar:
    member = tar.getmember("model.joblib")

    if not member.isfile():
        raise ValueError("model.joblib is not a regular file")

    model_file = tar.extractfile(member)

    if model_file is None:
        raise ValueError("Cannot read model.joblib")

    model = joblib.load(model_file)

# Load test dataset
df = pd.read_csv(test_path)
X_test = df.drop(columns=["target"])
y_test = df["target"]

# Generate predictions
y_pred = model.predict(X_test)

# Malignant = 0, Benign = 1
POS_LABEL = 0

accuracy = float(accuracy_score(y_test, y_pred))
precision = float(
    precision_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0)
)
recall = float(
    recall_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0)
)
f1 = float(
    f1_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0)
)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

report = {
    "positive_class": 0,
    "positive_class_name": "malignant",
    "test_samples": int(len(df)),
    "metrics": {
        "accuracy": {"value": accuracy},
        "precision": {"value": precision},
        "recall": {"value": recall},
        "f1": {"value": f1},
    },
    "confusion_matrix": {
        "labels": [0, 1],
        "matrix": cm.tolist(),
        "malignant_false_negatives": int(cm[0, 1]),
    },
}

# Save evaluation report
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "evaluation.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("=== Evaluation Completed ===")
print("Test samples:", len(df))
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix [0, 1]:")
print(cm)
print("Malignant false negatives:", int(cm[0, 1]))
print("Evaluation report:", output_path)
```

### Original cell 09

```python

import json
import os
import subprocess
import sys
import tarfile
from pathlib import Path

test_base = Path("local_test").resolve()
model_dir = test_base / "model"
model_path = model_dir / "model.joblib"
archive_path = model_dir / "model.tar.gz"

assert model_path.exists()
assert (test_base / "test" / "test.csv").exists()

# Simulate SageMaker Training model artifact
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(model_path, arcname="model.joblib")

# Run evaluation script
env = os.environ.copy()
env["PROCESSING_BASE"] = str(test_base)

result = subprocess.run(
    [sys.executable, "src/evaluate.py"],
    env=env,
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout)

# Validate evaluation.json
report_path = test_base / "evaluation" / "evaluation.json"

with open(report_path, encoding="utf-8") as f:
    report = json.load(f)

assert report["positive_class"] == 0
assert report["test_samples"] == 114
assert sum(
    sum(row)
    for row in report["confusion_matrix"]["matrix"]
) == 114

for metric in ["accuracy", "precision", "recall", "f1"]:
    value = report["metrics"][metric]["value"]
    assert 0.0 <= value <= 1.0

print("\n=== Validation ===")
print("Evaluation JSON exists:", report_path.exists())
print("Positive class:", report["positive_class"])
print("Evaluation validation: OK")
```

### Original cell 10

```python

from pathlib import Path
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core import image_uris

# Project configuration
REGION = "ap-northeast-2"
BUCKET = "REDACTED_BUCKET"
PREFIX = "breast-cancer-pipeline"

INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"

INPUT_URI = f"s3://{BUCKET}/{PREFIX}/input/breast_cancer.csv"

# SageMaker sessions
boto_session = boto3.Session(region_name=REGION)

sm_session = Session(
    boto_session=boto_session,
    default_bucket=BUCKET,
)

pipeline_session = PipelineSession(
    boto_session=boto_session,
    default_bucket=BUCKET,
)

role = get_execution_role()

# Scikit-learn container image
sklearn_image = image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version=FRAMEWORK_VERSION,
    py_version="py3",
    instance_type=INSTANCE_TYPE,
)

# Validate local source files
for filename in [
    "src/preprocess.py",
    "src/train.py",
    "src/evaluate.py",
]:
    assert Path(filename).is_file(), f"Missing: {filename}"

# Validate input S3 object
s3 = boto_session.client("s3")
s3.head_object(
    Bucket=BUCKET,
    Key=f"{PREFIX}/input/breast_cancer.csv",
)

assert sm_session.default_bucket() == BUCKET
assert pipeline_session.default_bucket() == BUCKET

print("=== Pipeline Configuration ===")
print("Region:", REGION)
print("S3 bucket:", BUCKET)
print("Input URI:", INPUT_URI)
print("Instance type:", INSTANCE_TYPE)
print("Framework:", FRAMEWORK_VERSION)
print("Image:", sklearn_image.split("/")[-1])
print("Execution role:", role.rsplit("/", 1)[-1])
print("Source scripts: OK")
print("Pipeline configuration: OK")
```

### Original cell 11

```python

from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
)
from sagemaker.core.workflow.functions import Join
from sagemaker.core.workflow.execution_variables import ExecutionVariables
from sagemaker.mlops.workflow.steps import ProcessingStep

# SageMaker Processing Job configuration
processor = ScriptProcessor(
    image_uri=sklearn_image,
    command=["python3"],
    role=role,
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    max_runtime_in_seconds=900,
    sagemaker_session=pipeline_session,
)

# Generate a unique S3 path for each pipeline execution
def processing_output_uri(output_name):
    return Join(
        on="/",
        values=[
            f"s3://{BUCKET}/{PREFIX}/runs",
            ExecutionVariables.PIPELINE_EXECUTION_ID,
            "preprocess",
            output_name,
        ],
    )

# Define the preprocessing step
prep_args = processor.run(
    code="src/preprocess.py",
    inputs=[
        ProcessingInput(
            input_name="breast-cancer-input",
            s3_input=ProcessingS3Input(
                s3_uri=INPUT_URI,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name=name,
            s3_output=ProcessingS3Output(
                s3_uri=processing_output_uri(name),
                local_path=f"/opt/ml/processing/{name}",
                s3_upload_mode="EndOfJob",
            ),
        )
        for name in ["train", "test"]
    ],
)

step_prep = ProcessingStep(
    name="Preprocess",
    step_args=prep_args,
)

print("=== Processing Step Configuration ===")
print("Step name:", step_prep.name)
print("Instance type:", INSTANCE_TYPE)
print("Input:", INPUT_URI)
print("Output names: train, test")
print("Preprocessing step definition: OK")
```

### Original cell 12

```python

from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    Compute,
    InputData,
    SourceCode,
    OutputDataConfig,
)
from sagemaker.mlops.workflow.steps import TrainingStep

# Pipeline 실행 시 변경할 하이퍼파라미터
n_estimators = ParameterString(
    name="NEstimators",
    default_value="10",
)

# 전처리 단계가 생성한 학습 데이터 참조
train_s3_uri = (
    step_prep.properties
    .ProcessingOutputConfig
    .Outputs["train"]
    .S3Output.S3Uri
)

# 모델 학습 작업 정의
trainer = ModelTrainer(
    training_image=sklearn_image,
    source_code=SourceCode(
        source_dir="src",
        entry_script="train.py",
    ),
    role=role,
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
    ),
    hyperparameters={
        "n_estimators": n_estimators,
        "random_state": "42",
    },
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=train_s3_uri,
            content_type="text/csv",
        )
    ],
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{BUCKET}/{PREFIX}/training"
    ),
    sagemaker_session=pipeline_session,
)

# PipelineSession에서 작업 정의 생성
train_args = trainer.train()

step_train = TrainingStep(
    name="Train",
    step_args=train_args,
)

print("=== Training Step Configuration ===")
print("Step name:", step_train.name)
print("Instance type:", INSTANCE_TYPE)
print("Hyperparameter:", n_estimators.name)
print("Default value:", n_estimators.default_value)
print("Model output: project S3 bucket")
print("Training step definition: OK")
```

### Original cell 13

```python

from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.mlops.workflow.steps import ProcessingStep

# TrainingStep에서 생성한 모델 아티팩트
model_s3_uri = (
    step_train.properties
    .ModelArtifacts.S3ModelArtifacts
)

# Preprocess에서 생성한 평가 데이터
test_s3_uri = (
    step_prep.properties
    .ProcessingOutputConfig
    .Outputs["test"]
    .S3Output.S3Uri
)

# SageMaker Pipeline에서 평가 JSON을 인식하도록 등록
eval_report = PropertyFile(
    name="EvalReport",
    output_name="evaluation",
    path="evaluation.json",
)

# 실행마다 독립적인 평가 결과 저장 경로
evaluation_s3_uri = Join(
    on="/",
    values=[
        f"s3://{BUCKET}/{PREFIX}/runs",
        ExecutionVariables.PIPELINE_EXECUTION_ID,
        "evaluate",
    ],
)

# Evaluate Processing Job 정의
eval_args = processor.run(
    code="src/evaluate.py",
    inputs=[
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                s3_uri=model_s3_uri,
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test",
            s3_input=ProcessingS3Input(
                s3_uri=test_s3_uri,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=evaluation_s3_uri,
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)

step_eval = ProcessingStep(
    name="Evaluate",
    step_args=eval_args,
    property_files=[eval_report],
)

print("=== Evaluation Step Configuration ===")
print("Step name:", step_eval.name)
print("Input sources: Train model + Preprocess test data")
print("Evaluation file:", eval_report.path)
print("Output: project S3 bucket / runs / execution ID")
print("Evaluation step definition: OK")
```

### Original cell 14

```python

import json

from sagemaker.mlops.workflow.pipeline import Pipeline

# Combine all three pipeline steps
pipeline = Pipeline(
    name="breast-cancer-ml-pipeline",
    parameters=[n_estimators],
    steps=[
        step_prep,
        step_train,
        step_eval,
    ],
    sagemaker_session=pipeline_session,
)

# Generate and validate the JSON definition
definition = json.loads(pipeline.definition())

# Verify structure
steps = definition["Steps"]
step_names = [step["Name"] for step in steps]
step_types = {step["Name"]: step["Type"] for step in steps}
parameter_names = [
    parameter["Name"]
    for parameter in definition.get("Parameters", [])
]

assert len(steps) == 3
assert step_names == ["Preprocess", "Train", "Evaluate"]
assert step_types["Preprocess"] == "Processing"
assert step_types["Train"] == "Training"
assert step_types["Evaluate"] == "Processing"
assert "NEstimators" in parameter_names

print("=== Pipeline Definition ===")
print("Pipeline name:", pipeline.name)
print("Steps:", step_names)
print("Step types:", step_types)
print("Parameters:", parameter_names)
print("JSON validation: OK")
print("Pipeline definition: OK")
```

### Original cell 15

```python

import boto3

# Create or update the SageMaker Pipeline
response = pipeline.upsert(
    role_arn=role,
    description=(
        "Breast cancer classification ML pipeline "
        "with preprocessing, training, and evaluation."
    ),
    tags=[
        {
            "Key": "Project",
            "Value": "breast-cancer-mlops",
        }
    ],
)

# Verify registration through AWS API
sm_client = boto3.client(
    "sagemaker",
    region_name=REGION,
)

pipeline_info = sm_client.describe_pipeline(
    PipelineName=pipeline.name
)

print("=== Pipeline Registration ===")
print("Pipeline name:", pipeline_info["PipelineName"])
print("Pipeline status:", pipeline_info.get("PipelineStatus"))
print("AWS registration: OK")
```

### Original cell 16

```python

import re
import boto3

# Student ID is used only in the execution display name.
student_id = input("학번을 입력하세요: ").strip()

assert re.fullmatch(r"[0-9]+", student_id), "학번은 숫자로 입력하세요."

run_name = f"{student_id}-RF-N10"

# Start the first pipeline execution.
execution1 = pipeline.start(
    parameters={
        "NEstimators": "10",
    },
    execution_display_name=run_name,
    execution_description="Random Forest experiment with 10 trees",
)

# Check execution status.
sm_client = boto3.client(
    "sagemaker",
    region_name=REGION,
)

execution_info = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution1.arn
)

print("=== Pipeline Execution 1 ===")
print("Execution name:", run_name)
print("NEstimators:", 10)
print("Status:", execution_info["PipelineExecutionStatus"])
```

### Original cell 17

```python

# Check the existing execution.
# Do NOT call pipeline.start() again.

info = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution1.arn
)

print("=== Pipeline Execution Status ===")
print("Status:", info["PipelineExecutionStatus"])

if info.get("FailureReason"):
    print("Failure reason:", info["FailureReason"])

response = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution1.arn
)

steps = {
    step["StepName"]: step
    for step in response.get("PipelineExecutionSteps", [])
}

print("\n=== Step Status ===")

for name in ["Preprocess", "Train", "Evaluate"]:
    step = steps.get(name)

    if step is None:
        print(f"{name}: Not started / not yet reported")
        continue

    print(f"{name}: {step['StepStatus']}")

    if step.get("FailureReason"):
        print("  Failure reason:", step["FailureReason"])
```

### Original cell 18

```python

import json
import boto3
from urllib.parse import urlparse

# 1. 첫 번째 실행의 Evaluate 단계 확인
response = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution1.arn
)

eval_step = next(
    step for step in response["PipelineExecutionSteps"]
    if step["StepName"] == "Evaluate"
)

assert eval_step["StepStatus"] == "Succeeded"

# 2. Evaluate Processing Job 조회
processing_arn = eval_step["Metadata"]["ProcessingJob"]["Arn"]
processing_name = processing_arn.rsplit("/", 1)[-1]

job = sm_client.describe_processing_job(
    ProcessingJobName=processing_name
)

# 3. evaluation 출력 경로 찾기
eval_output = next(
    output
    for output in job["ProcessingOutputConfig"]["Outputs"]
    if output["OutputName"] == "evaluation"
)

output_uri = eval_output["S3Output"]["S3Uri"]
parsed = urlparse(output_uri)

bucket_name = parsed.netloc
prefix = parsed.path.lstrip("/").rstrip("/") + "/"

s3 = boto3.client("s3", region_name=REGION)

objects = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=prefix
)

# 4. evaluation.json 파일 찾기
matches = [
    item["Key"]
    for item in objects.get("Contents", [])
    if item["Key"].endswith("evaluation.json")
]

assert len(matches) == 1, (
    f"Expected one evaluation.json, found {len(matches)}"
)

report_key = matches[0]

# 5. 평가 JSON 읽기
obj = s3.get_object(
    Bucket=bucket_name,
    Key=report_key
)

report = json.loads(obj["Body"].read())

# 6. 실제 평가 결과 출력
print("=== SageMaker Evaluation Result ===")
print("Execution:", "RF-N10")
print("Test samples:", report["test_samples"])
print("Positive class:", report["positive_class_name"])

for name in ["accuracy", "precision", "recall", "f1"]:
    value = report["metrics"][name]["value"]
    print(f"{name.capitalize():10s}: {value:.4f}")

print("\nConfusion matrix:")
for row in report["confusion_matrix"]["matrix"]:
    print(row)

print(
    "Malignant false negatives:",
    report["confusion_matrix"]["malignant_false_negatives"]
)

print("\nEvaluation JSON S3 URI:")
print(f"s3://{bucket_name}/{report_key}")
print("S3 evaluation verification: OK")
```

### Original cell 19

```python

# STEP 5-5: Second experiment
# Run this cell only once.

execution2 = pipeline.start(
    parameters={
        "NEstimators": "50",
    },
    execution_display_name="REDACTED-RF-N50",
    execution_description="Random Forest experiment with 50 trees",
)

info2 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution2.arn
)

print("=== Pipeline Execution 2 ===")
print("Execution name:", "REDACTED-RF-N50")
print("NEstimators:", 50)
print("Status:", info2["PipelineExecutionStatus"])
```

### Original cell 20

```python

# Check the second execution without starting another job.

info = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution2.arn
)

print("=== Pipeline Execution 2 Status ===")
print("Status:", info["PipelineExecutionStatus"])

if info.get("FailureReason"):
    print("Failure reason:", info["FailureReason"])

response = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution2.arn
)

steps = {
    step["StepName"]: step
    for step in response.get("PipelineExecutionSteps", [])
}

print("\n=== Step Status ===")

for name in ["Preprocess", "Train", "Evaluate"]:
    step = steps.get(name)

    if step is None:
        print(f"{name}: Not started / not yet reported")
    else:
        print(f"{name}: {step['StepStatus']}")

        if step.get("FailureReason"):
            print("  Failure reason:", step["FailureReason"])
```

### Original cell 21

```python

import json
from urllib.parse import urlparse

# 1. 두 번째 실행의 Evaluate 단계 조회
response2 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution2.arn
)

eval_step2 = next(
    step for step in response2["PipelineExecutionSteps"]
    if step["StepName"] == "Evaluate"
)

assert eval_step2["StepStatus"] == "Succeeded"

# 2. 실제 Evaluate Processing Job 확인
processing_arn2 = eval_step2["Metadata"]["ProcessingJob"]["Arn"]
processing_name2 = processing_arn2.rsplit("/", 1)[-1]

job2 = sm_client.describe_processing_job(
    ProcessingJobName=processing_name2
)

# 3. 평가 결과의 S3 경로 확인
eval_output2 = next(
    output
    for output in job2["ProcessingOutputConfig"]["Outputs"]
    if output["OutputName"] == "evaluation"
)

output_uri2 = eval_output2["S3Output"]["S3Uri"]
parsed2 = urlparse(output_uri2)

bucket2 = parsed2.netloc
prefix2 = parsed2.path.lstrip("/").rstrip("/") + "/"

objects2 = s3.list_objects_v2(
    Bucket=bucket2,
    Prefix=prefix2
)

matches2 = [
    item["Key"]
    for item in objects2.get("Contents", [])
    if item["Key"].endswith("evaluation.json")
]

assert len(matches2) == 1

# 4. 평가 JSON 읽기
report_key2 = matches2[0]

obj2 = s3.get_object(
    Bucket=bucket2,
    Key=report_key2
)

report_n50 = json.loads(obj2["Body"].read())

# 5. 평가 결과 출력
print("=== SageMaker Evaluation Result ===")
print("Execution: RF-N50")
print("NEstimators: 50")
print("Test samples:", report_n50["test_samples"])
print("Positive class:", "malignant")

for name in ["accuracy", "precision", "recall", "f1"]:
    value = report_n50["metrics"][name]["value"]
    print(f"{name.capitalize():10s}: {value:.4f}")

print("\nConfusion matrix:")
for row in report_n50["confusion_matrix"]["matrix"]:
    print(row)

print(
    "Malignant false negatives:",
    report_n50["confusion_matrix"]["malignant_false_negatives"]
)

evaluation_uri_n50 = f"s3://{bucket2}/{report_key2}"

print("\nEvaluation JSON S3 URI:")
print(evaluation_uri_n50)
print("S3 evaluation verification: OK")
```

### Original cell 22

```python

# STEP 5-8: Third experiment
# Run this cell only once.

execution3 = pipeline.start(
    parameters={
        "NEstimators": "100",
    },
    execution_display_name="REDACTED-RF-N100",
    execution_description="Random Forest experiment with 100 trees",
)

info3 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution3.arn
)

print("=== Pipeline Execution 3 ===")
print("Execution name:", "REDACTED-RF-N100")
print("NEstimators:", 100)
print("Status:", info3["PipelineExecutionStatus"])
```

### Original cell 23

```python

# STEP 5-9: Check the third execution.
# This code does NOT start a new job.

info3 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution3.arn
)

print("=== Pipeline Execution 3 Status ===")
print("Status:", info3["PipelineExecutionStatus"])

if info3.get("FailureReason"):
    print("Failure reason:", info3["FailureReason"])

response3 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution3.arn
)

steps3 = {
    step["StepName"]: step
    for step in response3.get("PipelineExecutionSteps", [])
}

print("\n=== Step Status ===")

for name in ["Preprocess", "Train", "Evaluate"]:
    step = steps3.get(name)

    if step is None:
        print(f"{name}: Not started / not yet reported")
    else:
        print(f"{name}: {step['StepStatus']}")

        if step.get("FailureReason"):
            print("  Failure reason:", step["FailureReason"])
```

### Original cell 24

```python

import json
from urllib.parse import urlparse

# 1. 세 번째 실행에서 Evaluate 단계 조회
response3 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution3.arn
)

eval_step3 = next(
    step for step in response3["PipelineExecutionSteps"]
    if step["StepName"] == "Evaluate"
)

assert eval_step3["StepStatus"] == "Succeeded"

# 2. 실제 Evaluate Processing Job 조회
processing_arn3 = eval_step3["Metadata"]["ProcessingJob"]["Arn"]
processing_name3 = processing_arn3.rsplit("/", 1)[-1]

job3 = sm_client.describe_processing_job(
    ProcessingJobName=processing_name3
)

# 3. S3 평가 결과 출력 경로 확인
eval_output3 = next(
    output
    for output in job3["ProcessingOutputConfig"]["Outputs"]
    if output["OutputName"] == "evaluation"
)

output_uri3 = eval_output3["S3Output"]["S3Uri"]
parsed3 = urlparse(output_uri3)

bucket3 = parsed3.netloc
prefix3 = parsed3.path.lstrip("/").rstrip("/") + "/"

objects3 = s3.list_objects_v2(
    Bucket=bucket3,
    Prefix=prefix3
)

matches3 = [
    item["Key"]
    for item in objects3.get("Contents", [])
    if item["Key"].endswith("evaluation.json")
]

assert len(matches3) == 1, (
    f"Expected one evaluation.json, found {len(matches3)}"
)

# 4. 실제 평가 JSON 읽기
report_key3 = matches3[0]

obj3 = s3.get_object(
    Bucket=bucket3,
    Key=report_key3
)

report_n100 = json.loads(obj3["Body"].read())

# 5. 실제 평가 지표 출력
print("=== SageMaker Evaluation Result ===")
print("Execution: RF-N100")
print("NEstimators: 100")
print("Test samples:", report_n100["test_samples"])
print("Positive class: malignant")

for name in ["accuracy", "precision", "recall", "f1"]:
    value = report_n100["metrics"][name]["value"]
    print(f"{name.capitalize():10s}: {value:.4f}")

print("\nConfusion matrix:")
for row in report_n100["confusion_matrix"]["matrix"]:
    print(row)

print(
    "Malignant false negatives:",
    report_n100["confusion_matrix"]["malignant_false_negatives"]
)

evaluation_uri_n100 = f"s3://{bucket3}/{report_key3}"

print("\nEvaluation JSON S3 URI:")
print(evaluation_uri_n100)
print("S3 evaluation verification: OK")
```

### Original cell 25

```python

from pathlib import Path
import json
import csv

# Local results directory
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

experiments = {
    10: report,
    50: report_n50,
    100: report_n100,
}

summary = []

for trees, evaluation in experiments.items():
    # Save each verified evaluation result
    output_file = results_dir / f"evaluation_n{trees}.json"

    output_file.write_text(
        json.dumps(evaluation, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    metrics = evaluation["metrics"]

    summary.append({
        "n_estimators": trees,
        "accuracy": metrics["accuracy"]["value"],
        "precision_malignant": metrics["precision"]["value"],
        "recall_malignant": metrics["recall"]["value"],
        "f1_malignant": metrics["f1"]["value"],
        "malignant_false_negatives": evaluation[
            "confusion_matrix"
        ]["malignant_false_negatives"],
    })

# Save comparison CSV
csv_path = results_dir / "experiment_comparison.csv"

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=list(summary[0].keys()),
    )
    writer.writeheader()
    writer.writerows(summary)

print("=== Experiment Results Saved ===")
for file in sorted(results_dir.glob("*")):
    print(file)

print("\n=== Experiment Comparison ===")
for row in summary:
    print(
        f"N={row['n_estimators']:3d} | "
        f"Accuracy={row['accuracy']:.4f} | "
        f"Precision={row['precision_malignant']:.4f} | "
        f"Recall={row['recall_malignant']:.4f} | "
        f"F1={row['f1_malignant']:.4f}"
    )

print("\nResults export: OK")
```

### Original cell 26

```python

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

project_dir = Path.cwd()
backup_path = project_dir / "breast-cancer-pipeline-private-backup.zip"

# Required project files
required_files = [
    "src/preprocess.py",
    "src/train.py",
    "src/evaluate.py",
    "results/evaluation_n10.json",
    "results/evaluation_n50.json",
    "results/evaluation_n100.json",
    "results/experiment_comparison.csv",
]

# Additional files, if available
optional_files = [
    "00_environment_check.ipynb",
    "01_pipeline_build.ipynb",
    "data/breast_cancer.csv",
    "docs/01-environment.md",
]

missing = [
    name for name in required_files
    if not (project_dir / name).is_file()
]

assert not missing, f"Missing required files: {missing}"

included = [
    name for name in required_files + optional_files
    if (project_dir / name).is_file()
]

with ZipFile(backup_path, "w", ZIP_DEFLATED) as archive:
    for name in included:
        archive.write(
            project_dir / name,
            arcname=name,
        )

print("=== Project Backup ===")
for name in included:
    print("Included:", name)

print("\nBackup file:", backup_path.name)
print("Backup size:", backup_path.stat().st_size, "bytes")
print("Project backup: OK")
```

### Original cell 27

```python

%%writefile src/train_multi.py
import argparse
import os
from pathlib import Path

import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# 1. Training parameters
parser = argparse.ArgumentParser()

parser.add_argument(
    "--model_type",
    type=str,
    choices=["random_forest", "logistic_regression"],
    default="random_forest",
)

parser.add_argument(
    "--use_scaling",
    type=str,
    choices=["true", "false"],
    default="false",
)

parser.add_argument(
    "--n_estimators",
    type=int,
    default=100,
)

parser.add_argument(
    "--random_state",
    type=int,
    default=42,
)

args = parser.parse_args()


# 2. SageMaker input/output paths
train_dir = Path(
    os.environ.get(
        "SM_CHANNEL_TRAIN",
        "/opt/ml/input/data/train",
    )
)

model_dir = Path(
    os.environ.get(
        "SM_MODEL_DIR",
        "/opt/ml/model",
    )
)

model_dir.mkdir(parents=True, exist_ok=True)


# 3. Load training data
df = pd.read_csv(train_dir / "train.csv")

X = df.drop(columns=["target"])
y = df["target"]


# 4. Select estimator
if args.model_type == "random_forest":
    estimator = RandomForestClassifier(
        n_estimators=args.n_estimators,
        random_state=args.random_state,
        n_jobs=-1,
    )

else:
    estimator = LogisticRegression(
        max_iter=2000,
        random_state=args.random_state,
    )


# 5. Optional feature scaling
if args.use_scaling == "true":
    model = make_pipeline(
        StandardScaler(),
        estimator,
    )
else:
    model = estimator


# 6. Train model
model.fit(X, y)


# 7. Save model
model_path = model_dir / "model.joblib"
joblib.dump(model, model_path)


# 8. Training summary
print("=== Multi-Model Training ===")
print("Model type:", args.model_type)
print("Use scaling:", args.use_scaling)
print("N estimators:", args.n_estimators)
print("Random state:", args.random_state)
print("Training samples:", len(X))
print("Features:", X.shape[1])
print("Model saved:", model_path)
print("Training script: OK")
```

### Original cell 28

```python

import os
import sys
import json
import tarfile
import tempfile
import subprocess
from pathlib import Path

project_dir = Path.cwd()

train_csv = project_dir / "local_test/train/train.csv"
test_csv = project_dir / "local_test/test/test.csv"

assert train_csv.is_file(), f"Missing: {train_csv}"
assert test_csv.is_file(), f"Missing: {test_csv}"

print("=== STEP 7-2: Local LR Validation ===")

for scaling in ["false", "true"]:
    label = "LR-NOSCALE" if scaling == "false" else "LR-SCALE"

    with tempfile.TemporaryDirectory() as temp:
        base = Path(temp)

        model_dir = base / "model"
        evaluation_dir = base / "evaluation"
        local_test_dir = base / "test"

        model_dir.mkdir()
        evaluation_dir.mkdir()
        local_test_dir.mkdir()

        # Training environment
        train_env = os.environ.copy()
        train_env["SM_CHANNEL_TRAIN"] = str(train_csv.parent)
        train_env["SM_MODEL_DIR"] = str(model_dir)

        train_result = subprocess.run(
            [
                sys.executable,
                str(project_dir / "src/train_multi.py"),
                "--model_type", "logistic_regression",
                "--use_scaling", scaling,
                "--random_state", "42",
            ],
            env=train_env,
            capture_output=True,
            text=True,
            check=True,
        )

        model_file = model_dir / "model.joblib"
        assert model_file.is_file()

        # Package model in SageMaker's tar.gz format
        with tarfile.open(
            model_dir / "model.tar.gz", "w:gz"
        ) as archive:
            archive.add(model_file, arcname="model.joblib")

        # Prepare evaluation test data
        import shutil
        shutil.copy2(test_csv, local_test_dir / "test.csv")

        # Evaluation environment
        eval_env = os.environ.copy()
        eval_env["PROCESSING_BASE"] = str(base)

        eval_result = subprocess.run(
            [
                sys.executable,
                str(project_dir / "src/evaluate.py"),
            ],
            env=eval_env,
            capture_output=True,
            text=True,
            check=True,
        )

        # Verify evaluation.json
        report_path = evaluation_dir / "evaluation.json"
        assert report_path.is_file()

        report_lr = json.loads(
            report_path.read_text(encoding="utf-8")
        )

        assert report_lr["positive_class"] == 0
        assert report_lr["test_samples"] == 114

        for metric in ["accuracy", "precision", "recall", "f1"]:
            assert metric in report_lr["metrics"]

        print(f"\n=== {label} ===")
        print("Training: OK")
        print("Model artifact: OK")
        print("Evaluation JSON: OK")
        print("Positive class: 0 (malignant)")

        for metric in ["accuracy", "precision", "recall", "f1"]:
            value = report_lr["metrics"][metric]["value"]
            print(f"{metric:10s}: {value:.4f}")

        print(
            "Malignant false negatives:",
            report_lr["confusion_matrix"][
                "malignant_false_negatives"
            ],
        )

print("\nLocal LR scaling comparison: OK")
```

### Original cell 29

```python

import json
from pathlib import Path

from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.core.workflow.functions import Join
from sagemaker.core.workflow.execution_variables import ExecutionVariables
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
)
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    Compute,
    InputData,
    SourceCode,
    OutputDataConfig,
)
from sagemaker.mlops.workflow.steps import (
    TrainingStep,
    ProcessingStep,
)
from sagemaker.mlops.workflow.pipeline import Pipeline

assert Path("src/train_multi.py").is_file()
assert Path("src/evaluate.py").is_file()

# 1. New pipeline parameters
model_type = ParameterString(
    name="ModelType",
    default_value="random_forest",
)

use_scaling = ParameterString(
    name="UseScaling",
    default_value="false",
)

# Reuse existing NEstimators parameter
# n_estimators was defined in STEP 4-3.

# 2. Reuse preprocessing training output
train_input_multi = (
    step_prep.properties
    .ProcessingOutputConfig
    .Outputs["train"]
    .S3Output.S3Uri
)

# 3. New multi-model trainer
trainer_multi = ModelTrainer(
    training_image=sklearn_image,
    source_code=SourceCode(
        source_dir="src",
        entry_script="train_multi.py",
    ),
    role=role,
    compute=Compute(
        instance_type=INSTANCE_TYPE,
        instance_count=1,
    ),
    hyperparameters={
        "model_type": model_type,
        "use_scaling": use_scaling,
        "n_estimators": n_estimators,
        "random_state": "42",
    },
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=train_input_multi,
            content_type="text/csv",
        ),
    ],
    output_data_config=OutputDataConfig(
        s3_output_path=f"s3://{BUCKET}/{PREFIX}/training"
    ),
    sagemaker_session=pipeline_session,
)

train_args_multi = trainer_multi.train()

step_train_multi = TrainingStep(
    name="Train",
    step_args=train_args_multi,
)

# 4. Reconnect Evaluate to the NEW TrainingStep
eval_report_multi = PropertyFile(
    name="EvalReport",
    output_name="evaluation",
    path="evaluation.json",
)

model_input_multi = (
    step_train_multi.properties
    .ModelArtifacts.S3ModelArtifacts
)

test_input_multi = (
    step_prep.properties
    .ProcessingOutputConfig
    .Outputs["test"]
    .S3Output.S3Uri
)

eval_args_multi = processor.run(
    code="src/evaluate.py",
    inputs=[
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                s3_uri=model_input_multi,
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test",
            s3_input=ProcessingS3Input(
                s3_uri=test_input_multi,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=Join(
                    on="/",
                    values=[
                        f"s3://{BUCKET}/{PREFIX}/runs",
                        ExecutionVariables.PIPELINE_EXECUTION_ID,
                        "evaluate",
                    ],
                ),
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
)

step_eval_multi = ProcessingStep(
    name="Evaluate",
    step_args=eval_args_multi,
    property_files=[eval_report_multi],
)

# 5. Build updated pipeline locally
# Same AWS pipeline name, but NOT registered yet.
pipeline_multi = Pipeline(
    name=pipeline.name,
    parameters=[
        n_estimators,
        model_type,
        use_scaling,
    ],
    steps=[
        step_prep,
        step_train_multi,
        step_eval_multi,
    ],
    sagemaker_session=pipeline_session,
)

# 6. Validate pipeline JSON
definition_multi = json.loads(
    pipeline_multi.definition()
)

param_names = [
    p["Name"]
    for p in definition_multi["Parameters"]
]

step_names = [
    s["Name"]
    for s in definition_multi["Steps"]
]

train_def = next(
    s for s in definition_multi["Steps"]
    if s["Name"] == "Train"
)

eval_def = next(
    s for s in definition_multi["Steps"]
    if s["Name"] == "Evaluate"
)

training_json = json.dumps(
    train_def["Arguments"]
)

evaluation_json = json.dumps(
    eval_def["Arguments"]
)

property_files = eval_def.get("PropertyFiles", [])

assert set(param_names) == {
    "NEstimators", "ModelType", "UseScaling"
}
assert step_names == [
    "Preprocess", "Train", "Evaluate"
]

assert "Parameters.ModelType" in training_json
assert "Parameters.UseScaling" in training_json
assert "Parameters.NEstimators" in training_json

assert "Steps.Train.ModelArtifacts" in evaluation_json
assert "Steps.Preprocess.ProcessingOutputConfig" in evaluation_json

assert any(
    p["Name"] == "EvalReport"
    and p["OutputName"] == "evaluation"
    and p["FilePath"] == "evaluation.json"
    for p in property_files
)

print("=== Multi-Model Pipeline Validation ===")
print("Pipeline:", pipeline_multi.name)
print("Steps:", step_names)
print("Parameters:", param_names)
print("Train parameter references: OK")
print("Evaluate dependencies: OK")
print("PropertyFile registration: OK")
print("Updated pipeline definition: OK")
```

### Original cell 30

```python

import json

# Use the pipeline definition already generated in STEP 7-3
eval_def = next(
    s for s in definition_multi["Steps"]
    if s["Name"] == "Evaluate"
)

property_files = eval_def.get("PropertyFiles", [])

print("=== Actual PropertyFiles JSON ===")
print(json.dumps(property_files, indent=2))

# Correct field name: PropertyFileName, not Name
assert any(
    p.get("PropertyFileName") == "EvalReport"
    and p.get("OutputName") == "evaluation"
    and p.get("FilePath") == "evaluation.json"
    for p in property_files
), "EvalReport PropertyFile configuration mismatch"

print("\n=== STEP 7-3 Final Validation ===")
print("Pipeline:", pipeline_multi.name)
print(
    "Steps:",
    [s["Name"] for s in definition_multi["Steps"]]
)
print(
    "Parameters:",
    [p["Name"] for p in definition_multi["Parameters"]]
)
print("Train parameter references: OK")
print("Evaluate dependencies: OK")
print("PropertyFile registration: OK")
print("Updated pipeline definition: OK")
```

### Original cell 31

```python

import json

print("=== STEP 7-4: Pipeline Update ===")

# 1. Verify existing successful executions
old_executions = {
    "RF-N10": execution1.arn,
    "RF-N50": execution2.arn,
    "RF-N100": execution3.arn,
}

for name, arn in old_executions.items():
    info = sm_client.describe_pipeline_execution(
        PipelineExecutionArn=arn
    )
    status = info["PipelineExecutionStatus"]
    print(f"Before update - {name}: {status}")
    assert status == "Succeeded"

# 2. Register updated pipeline definition
# This does NOT start a new pipeline execution.
pipeline_multi.upsert(
    role_arn=role,
    description=(
        "Breast cancer classification pipeline "
        "supporting Random Forest, Logistic Regression, "
        "and optional feature scaling."
    ),
)

# 3. Verify definition registered in AWS
aws_info = sm_client.describe_pipeline(
    PipelineName=pipeline_multi.name
)

aws_definition = json.loads(
    aws_info["PipelineDefinition"]
)

aws_parameters = [
    p["Name"]
    for p in aws_definition["Parameters"]
]

aws_steps = [
    s["Name"]
    for s in aws_definition["Steps"]
]

assert set(aws_parameters) == {
    "NEstimators",
    "ModelType",
    "UseScaling",
}
assert aws_steps == [
    "Preprocess",
    "Train",
    "Evaluate",
]

aws_eval = next(
    step for step in aws_definition["Steps"]
    if step["Name"] == "Evaluate"
)

assert any(
    p.get("PropertyFileName") == "EvalReport"
    and p.get("OutputName") == "evaluation"
    and p.get("FilePath") == "evaluation.json"
    for p in aws_eval.get("PropertyFiles", [])
)

# 4. Recheck previous executions after update
print("\n=== Previous Execution Verification ===")

for name, arn in old_executions.items():
    info = sm_client.describe_pipeline_execution(
        PipelineExecutionArn=arn
    )
    status = info["PipelineExecutionStatus"]
    print(f"{name}: {status}")
    assert status == "Succeeded"

print("\n=== Updated Pipeline ===")
print("Pipeline name:", aws_info["PipelineName"])
print("Pipeline status:", aws_info["PipelineStatus"])
print("AWS parameters:", aws_parameters)
print("AWS steps:", aws_steps)
print("PropertyFile: OK")
print("Existing executions preserved: OK")
print("Pipeline update: OK")
```

### Original cell 32

```python

# STEP 7-5: Logistic Regression without scaling
# Run this cell ONLY ONCE.

execution4 = pipeline_multi.start(
    parameters={
        "NEstimators": "10",
        "ModelType": "logistic_regression",
        "UseScaling": "false",
    },
    execution_display_name="REDACTED-LR-NOSCALE",
    execution_description=(
        "Logistic Regression experiment "
        "without feature scaling"
    ),
)

# Check the new execution status
info4 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution4.arn
)

print("=== Pipeline Execution 4 ===")
print("Execution name:", "REDACTED-LR-NOSCALE")
print("Model type:", "logistic_regression")
print("Use scaling:", "false")
print("Status:", info4["PipelineExecutionStatus"])
```

### Original cell 33

```python

# STEP 7-5-1: Check LR-NOSCALE execution
# This code only checks status. It does not create a new job.

info4 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution4.arn
)

print("=== LR-NOSCALE Pipeline Status ===")
print("Execution status:", info4["PipelineExecutionStatus"])

if info4.get("FailureReason"):
    print("Failure reason:", info4["FailureReason"])

response4 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution4.arn
)

steps4 = {
    step["StepName"]: step
    for step in response4.get("PipelineExecutionSteps", [])
}

print("\n=== Step Status ===")

for name in ["Preprocess", "Train", "Evaluate"]:
    step = steps4.get(name)

    if step is None:
        print(f"{name}: Not started / not yet reported")
        continue

    print(f"{name}: {step['StepStatus']}")

    if step.get("FailureReason"):
        print("  Failure reason:", step["FailureReason"])
```

### Original cell 34

```python

import json
import boto3
from urllib.parse import urlparse

# 1. Find Evaluate step from execution 4
response4 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution4.arn
)

eval_step4 = next(
    step for step in response4["PipelineExecutionSteps"]
    if step["StepName"] == "Evaluate"
)

assert eval_step4["StepStatus"] == "Succeeded"

# 2. Find the actual Processing Job
processing_arn4 = eval_step4["Metadata"]["ProcessingJob"]["Arn"]
processing_name4 = processing_arn4.rsplit("/", 1)[-1]

job4 = sm_client.describe_processing_job(
    ProcessingJobName=processing_name4
)

# 3. Find evaluation output in S3
eval_output4 = next(
    output
    for output in job4["ProcessingOutputConfig"]["Outputs"]
    if output["OutputName"] == "evaluation"
)

output_uri4 = eval_output4["S3Output"]["S3Uri"]
parsed4 = urlparse(output_uri4)

bucket4 = parsed4.netloc
prefix4 = parsed4.path.lstrip("/").rstrip("/") + "/"

s3 = boto3.client("s3", region_name=REGION)

objects4 = s3.list_objects_v2(
    Bucket=bucket4,
    Prefix=prefix4
)

matches4 = [
    item["Key"]
    for item in objects4.get("Contents", [])
    if item["Key"].endswith("evaluation.json")
]

assert len(matches4) == 1, (
    f"Expected one evaluation.json, found {len(matches4)}"
)

# 4. Read the actual evaluation JSON
report_key4 = matches4[0]

obj4 = s3.get_object(
    Bucket=bucket4,
    Key=report_key4
)

report_lr_noscale = json.loads(obj4["Body"].read())

# 5. Validate required assignment conditions
assert report_lr_noscale["positive_class"] == 0
assert report_lr_noscale["test_samples"] == 114

for metric in ["accuracy", "precision", "recall", "f1"]:
    assert metric in report_lr_noscale["metrics"]

# 6. Print evaluation results
print("=== SageMaker LR-NOSCALE Evaluation ===")
print("Model: Logistic Regression")
print("Scaling: OFF")
print("Test samples:", report_lr_noscale["test_samples"])
print("Positive class: 0 (malignant)")

for metric in ["accuracy", "precision", "recall", "f1"]:
    value = report_lr_noscale["metrics"][metric]["value"]
    print(f"{metric.capitalize():10s}: {value:.4f}")

print("\nConfusion matrix:")
for row in report_lr_noscale["confusion_matrix"]["matrix"]:
    print(row)

print(
    "Malignant false negatives:",
    report_lr_noscale["confusion_matrix"][
        "malignant_false_negatives"
    ]
)

evaluation_uri_lr_noscale = f"s3://{bucket4}/{report_key4}"

print("\nEvaluation JSON S3 URI:")
print(evaluation_uri_lr_noscale)
print("S3 evaluation verification: OK")
```

### Original cell 35

```python

# STEP 7-6: Logistic Regression with scaling
# Run this cell ONLY ONCE.

execution5 = pipeline_multi.start(
    parameters={
        "NEstimators": "10",
        "ModelType": "logistic_regression",
        "UseScaling": "true",
    },
    execution_display_name="REDACTED-LR-SCALE",
    execution_description=(
        "Logistic Regression experiment "
        "with StandardScaler"
    ),
)

# Check the fifth pipeline execution
info5 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution5.arn
)

print("=== Pipeline Execution 5 ===")
print("Execution name:", "REDACTED-LR-SCALE")
print("Model type:", "logistic_regression")
print("Use scaling:", "true")
print("Status:", info5["PipelineExecutionStatus"])
```

### Original cell 36

```python

# STEP 7-6-1: Check LR-SCALE execution status
# This code does NOT start a new pipeline execution.

info5 = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution5.arn
)

print("=== LR-SCALE Pipeline Status ===")
print("Execution status:", info5["PipelineExecutionStatus"])

if info5.get("FailureReason"):
    print("Failure reason:", info5["FailureReason"])

response5 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution5.arn
)

steps5 = {
    step["StepName"]: step
    for step in response5.get("PipelineExecutionSteps", [])
}

print("\n=== Step Status ===")

for name in ["Preprocess", "Train", "Evaluate"]:
    step = steps5.get(name)

    if step is None:
        print(f"{name}: Not started / not yet reported")
        continue

    print(f"{name}: {step['StepStatus']}")

    if step.get("FailureReason"):
        print("  Failure reason:", step["FailureReason"])
```

### Original cell 37

```python

import json
import boto3
from urllib.parse import urlparse

# 1. Find Evaluate step for LR-SCALE
response5 = sm_client.list_pipeline_execution_steps(
    PipelineExecutionArn=execution5.arn
)

eval_step5 = next(
    step for step in response5["PipelineExecutionSteps"]
    if step["StepName"] == "Evaluate"
)

assert eval_step5["StepStatus"] == "Succeeded"

# 2. Find actual Evaluate Processing Job
processing_arn5 = eval_step5["Metadata"]["ProcessingJob"]["Arn"]
processing_name5 = processing_arn5.rsplit("/", 1)[-1]

job5 = sm_client.describe_processing_job(
    ProcessingJobName=processing_name5
)

# 3. Find evaluation.json in S3
eval_output5 = next(
    output
    for output in job5["ProcessingOutputConfig"]["Outputs"]
    if output["OutputName"] == "evaluation"
)

output_uri5 = eval_output5["S3Output"]["S3Uri"]
parsed5 = urlparse(output_uri5)

bucket5 = parsed5.netloc
prefix5 = parsed5.path.lstrip("/").rstrip("/") + "/"

s3 = boto3.client("s3", region_name=REGION)

objects5 = s3.list_objects_v2(
    Bucket=bucket5,
    Prefix=prefix5
)

matches5 = [
    item["Key"]
    for item in objects5.get("Contents", [])
    if item["Key"].endswith("evaluation.json")
]

assert len(matches5) == 1, (
    f"Expected one evaluation.json, found {len(matches5)}"
)

# 4. Read the actual evaluation report
report_key5 = matches5[0]

obj5 = s3.get_object(
    Bucket=bucket5,
    Key=report_key5
)

report_lr_scale = json.loads(obj5["Body"].read())

# 5. Verify assignment requirements
assert report_lr_scale["positive_class"] == 0
assert report_lr_scale["test_samples"] == 114

for metric in ["accuracy", "precision", "recall", "f1"]:
    assert metric in report_lr_scale["metrics"]

# 6. Print verified evaluation metrics
print("=== SageMaker LR-SCALE Evaluation ===")
print("Model: Logistic Regression")
print("Scaling: ON")
print("Test samples:", report_lr_scale["test_samples"])
print("Positive class: 0 (malignant)")

for metric in ["accuracy", "precision", "recall", "f1"]:
    value = report_lr_scale["metrics"][metric]["value"]
    print(f"{metric.capitalize():10s}: {value:.4f}")

print("\nConfusion matrix:")
for row in report_lr_scale["confusion_matrix"]["matrix"]:
    print(row)

print(
    "Malignant false negatives:",
    report_lr_scale["confusion_matrix"][
        "malignant_false_negatives"
    ]
)

evaluation_uri_lr_scale = f"s3://{bucket5}/{report_key5}"

print("\nEvaluation JSON S3 URI:")
print(evaluation_uri_lr_scale)
print("S3 evaluation verification: OK")
```

### Original cell 38

```python

from pathlib import Path
import json
import pandas as pd

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

experiments = [
    ("RF-N10", execution1, report,
     "RandomForest", "OFF", 10),
    ("RF-N50", execution2, report_n50,
     "RandomForest", "OFF", 50),
    ("RF-N100", execution3, report_n100,
     "RandomForest", "OFF", 100),
    ("LR-NOSCALE", execution4, report_lr_noscale,
     "LogisticRegression", "OFF", None),
    ("LR-SCALE", execution5, report_lr_scale,
     "LogisticRegression", "ON", None),
]

rows = []

print("=== Five SageMaker Pipeline Executions ===")

for label, execution, evaluation, model, scaling, n in experiments:
    info = sm_client.describe_pipeline_execution(
        PipelineExecutionArn=execution.arn
    )

    assert info["PipelineExecutionStatus"] == "Succeeded"
    assert evaluation["positive_class"] == 0
    assert evaluation["test_samples"] == 114

    display_name = info.get("PipelineExecutionDisplayName")
    assert display_name, "Execution display name not returned"

    metrics = evaluation["metrics"]
    assert all(
        name in metrics
        for name in ["accuracy", "precision", "recall", "f1"]
    )

    # Save the verified evaluation JSON locally
    json_path = results_dir / f"evaluation_{label.lower()}.json"
    json_path.write_text(
        json.dumps(evaluation, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    rows.append({
        "Experiment": label,
        "Model": model,
        "Scaling": scaling,
        "NEstimators": n,
        "Accuracy": metrics["accuracy"]["value"],
        "Precision": metrics["precision"]["value"],
        "Recall": metrics["recall"]["value"],
        "F1": metrics["f1"]["value"],
        "Malignant_FN": evaluation["confusion_matrix"][
            "malignant_false_negatives"
        ],
    })

    # Student ID appears in console evidence only,
    # not in the public comparison CSV.
    print(display_name, "|", info["PipelineExecutionStatus"])

df = pd.DataFrame(rows)

# Save a comparison CSV without student IDs or AWS account details
csv_path = results_dir / "experiment_comparison_5runs.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\n=== Five-Run Evaluation Comparison ===")
print(
    df.to_string(
        index=False,
        formatters={
            metric: "{:.4f}".format
            for metric in ["Accuracy", "Precision", "Recall", "F1"]
        },
        na_rep="-",
    )
)

print("\nPositive class: 0 (malignant)")
print("Test samples per experiment: 114")
print("Comparison CSV:", csv_path)
print("Five-run comparison: OK")
```

### Original cell 39

```python

import json

# Read the actual evaluation.json from AWS S3
obj = s3.get_object(
    Bucket=bucket5,
    Key=report_key5,
)

evaluation_data = json.loads(
    obj["Body"].read().decode("utf-8")
)

assert evaluation_data["positive_class"] == 0
assert evaluation_data["test_samples"] == 114

print("=== AWS S3 Evaluation JSON ===")
print("Execution: REDACTED-LR-SCALE")
print("S3 URI:", evaluation_uri_lr_scale)

print("\n=== evaluation.json Contents ===")
print(json.dumps(
    evaluation_data,
    indent=2,
    ensure_ascii=False,
))

print("\nS3 JSON content verification: OK")
```

### Original cell 40

```python

import json

# Existing verified S3 evaluation files
evaluation_files = {
    "RF-N10": (
        execution1,
        bucket_name,
        report_key,
    ),
    "RF-N50": (
        execution2,
        bucket2,
        report_key2,
    ),
    "RF-N100": (
        execution3,
        bucket3,
        report_key3,
    ),
    "LR-NOSCALE": (
        execution4,
        bucket4,
        report_key4,
    ),
    "LR-SCALE": (
        execution5,
        bucket5,
        report_key5,
    ),
}

# Change this value to view each experiment
selected_experiment = "LR-SCALE"

execution, bucket, key = evaluation_files[
    selected_experiment
]

# Verify actual AWS execution
info = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution.arn
)

assert info["PipelineExecutionStatus"] == "Succeeded"

# Read evaluation.json directly from S3
obj = s3.get_object(
    Bucket=bucket,
    Key=key,
)

evaluation_data = json.loads(
    obj["Body"].read().decode("utf-8")
)

assert evaluation_data["positive_class"] == 0
assert evaluation_data["test_samples"] == 114
assert all(
    metric in evaluation_data["metrics"]
    for metric in ["accuracy", "precision", "recall", "f1"]
)

print("=== AWS SageMaker Evaluation Evidence ===")
print(
    "Execution:",
    info["PipelineExecutionDisplayName"]
)
print("Status:", info["PipelineExecutionStatus"])
print("S3 URI:", f"s3://{bucket}/{key}")

print("\n=== evaluation.json Contents ===")
print(
    json.dumps(
        evaluation_data,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nEvaluation evidence verification: OK")
```